# Kredi Kartı Dolandırıcılığı Tespiti

Önceki egzersizler, bir sinir ağının tüm farklı bileşenlerine daha yakından bakmanızı sağladı:
* sıralı (sequential) yoğun (Dense) bir Sinir Ağı mimarisi,
* derleme (compilation) yöntemi,
* modelin eğitilmesi (fitting).

Şimdi **çok fazla veri içeren** gerçek hayat bir veri seti üzerinde çalışalım!

**Veri seti: `Kredi Kartı İşlemleri (Credit Card Transactions)`**

Bu açık uçlu challenge’da, **kredi kartı işlemlerinden elde edilmiş verilerle** çalışacaksınız.

Bu veriler **hassas** olduğu için, toplam 31 sütundan yalnızca 3’ü bilinmektedir; geri kalanlar verileri **anonimleştirmek** amacıyla dönüştürülmüştür (aslında bunlar, orijinal verilerin **PCA projeksiyonlarıdır**).

Bilinen 3 sütun şunlardır:

* `TIME`: İşlemin, veri setindeki ilk işleme göre geçen süresi  
* `AMOUNT`: İşlem tutarı  
* `CLASS` (hedef değişkenimiz):
    * `0 : geçerli işlem`
    * `1 : sahte (fraud) işlem`

❓ **Soru** ❓ Veri setini indirerek başlayın:
* Kaggle üzerinden: [buradan](https://www.kaggle.com/mlg-ulb/creditcardfraud)
* veya bizim URL’mizden: [buradan](https://d32aokrjazspmn.cloudfront.net/materials/creditcard.csv)

Veriyi yükleyerek `X` ve `y` değişkenlerini oluşturun.

In [2]:
import pandas as pd

# 1. Veriyi yerel klasörden yükleyelim
df = pd.read_csv('creditcard.csv')

# 2. X ve y değişkenlerini oluşturalım
# Hedef değişkenimiz (y): Sadece 'Class' sütunu
y = df['Class']

# Özelliklerimiz (X): 'Class' sütunu dışındaki tüm sütunlar
X = df.drop(columns=['Class'])

# Sonuçları kontrol edelim
print(f"✅ Veri seti yüklendi: {df.shape[0]} işlem bulundu.")
print(f"✅ X (Özellikler) boyutu: {X.shape}")
print(f"✅ y (Hedef) boyutu: {y.shape}")

# Verinin ilk 5 satırına hızlıca bir bakış
X.head()

✅ Veri seti yüklendi: 284807 işlem bulundu.
✅ X (Özellikler) boyutu: (284807, 30)
✅ y (Hedef) boyutu: (284807,)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,0.251412,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.069083,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.524980,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.208038,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,0.408542,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99


## 1. Sınıfların yeniden dengelenmesi

In [3]:
# Sınıf dengesini kontrol edelim
pd.Series(y).value_counts(normalize=True)

Class
0    0.998273
1    0.001727
Name: proportion, dtype: float64

☝️ Bu `fraud detection` (sahtekârlık tespiti) challenge’ında **sınıflar aşırı derecede dengesizdir**:
* %99.8 normal işlemler
* %0.2 sahte (fraud) işlemler

**Ciddi yeniden dengeleme (rebalancing) stratejileri uygulamazsak sahtekârlık vakalarını tespit edemeyiz!**

❓ **Soru** ❓
1. **Önce**, veri setinizden üç ayrı bölme oluşturun: `Train / Val / Test`.  
   Doğrulama (validation) ve test setlerinin **dengesiz** kalması son derece önemlidir; böylece modeli değerlendirirken gerçek koşullar korunur ve veri sızıntısı (data leakage) oluşmaz. Test setinizi bu notebook’un **en son hücresine kadar saklayın**!

&nbsp;

2. **İkinci olarak**, yalnızca eğitim setinizi (training set) yeniden dengeleyin. Birçok seçeneğiniz var:

- Azınlık sınıfını rastgele oversample etmek (düz NumPy fonksiyonlarıyla).  
  *(En iyi seçenek değildir; çünkü satırları kopyalayarak veri sızıntısı yaratır.)*
- <a href="https://machinelearningmastery.com/smote-oversampling-for-imbalanced-classification/">**`Synthetic Minority Oversampling Technique - SMOTE`**</a> kullanarak, mevcut gözlemleri ağırlıklandırıp yeni veri noktaları üretmek
- Buna ek olarak, çoğunluk sınıfını bir miktar küçültmek için  
  <a href="https://machinelearningmastery.com/random-oversampling-and-undersampling-for-imbalanced-classification/">**`RandomUnderSampler`**</a> da deneyebilirsiniz

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline

# 1. ADIM: Üçlü Bölme (Train %70, Val %15, Test %15)
# Önce %70 Eğitim, %30 Geçici (Temp)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Kalan %30'u ikiye bölüyoruz (Validation ve Test)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# 2. ADIM: Ölçeklendirme (Scaling)
# SMOTE, en yakın komşulara (KNN) baktığı için verilerin ölçekli olması şarttır.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# 3. ADIM: Yeniden Dengeleme (Sadece Training Setine!)
# SMOTE ile azınlık sınıfını %10'a kadar artıralım
# RandomUnderSampler ile çoğunluk sınıfını biraz azaltıp denge kuralım
over = SMOTE(sampling_strategy=0.1, random_state=42)
under = RandomUnderSampler(sampling_strategy=0.5, random_state=42)

# Pipeline kullanarak sırayla uygula
steps = [('o', over), ('u', under)]
pipeline = Pipeline(steps=steps)

X_train_res, y_train_res = pipeline.fit_resample(X_train_scaled, y_train)

# SONUÇLARI KONTROL ET
print(f"--- Bölme Sonrası Boyutlar ---")
print(f"Eğitim: {X_train_res.shape[0]} | Doğrulama: {X_val_scaled.shape[0]} | Test: {X_test_scaled.shape[0]}")
print(f"\nYeni Eğitim Setindeki Dağılım:")
print(y_train_res.value_counts())

--- Bölme Sonrası Boyutlar ---
Eğitim: 59706 | Doğrulama: 42721 | Test: 42722

Yeni Eğitim Setindeki Dağılım:
Class
0    39804
1    19902
Name: count, dtype: int64


## 2. Sinir Ağı yinelemeleri

Sınıflarınızı yeniden dengelediğinize göre, test puanınızı optimize etmek için bir sinir ağı uyarlayın. Aşağıdaki ipuçlarını kullanmaktan çekinmeyin:

- Girişlerinizi normalleştirin!
    - Modelinizdeki ön işlemeyi “boru hattı” haline getirmek için tercihen model içinde bir [`Normalization`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Normalization) katmanı kullanın. 
    - Veya modelinizin dışında sklearn'in [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) öğesini kullanın, `X_train`, `X_val` ve `X_test` öğelerini uygulayın.
- Modelinizi aşırı uyumlu hale getirin, ardından aşağıdakileri kullanarak düzenleyin:
- Erken Durdurma kriterleri
- [`Dropout`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dropout) katmanları
    - veya [`düzenleyiciler`](https://www.tensorflow.org/api_docs/python/tf/keras/regularizers) katmanları
- 🚨 İzlemek istediğiniz metrikleri ve kullanmak istediğiniz kayıp fonksiyonunu dikkatlice düşünün!

In [6]:
import os
import warnings
import tensorflow as tf
from tensorflow.keras import models, layers, metrics, callbacks, Input

# 1. GEREKSİZ UYARILARI SUSTURMA
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' 
warnings.filterwarnings('ignore')

# 2. MODEL MİMARİSİ (Güncel 'Input' yöntemiyle)
model = models.Sequential([
    Input(shape=(X_train_res.shape[1],)), # Giriş katmanı
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),                   # Aşırı öğrenme (overfit) koruması
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')  # Çıkış: 0-1 arası olasılık
])

# 3. MODELİ DERLEME (Hırsız avcısı metrikleri)
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        metrics.Recall(name='recall'),       # Hırsız yakalama gücü
        metrics.Precision(name='precision'), # Yanlış alarm oranı
        metrics.AUC(name='auc', curve='PR')  # Dengesiz veri başarı puanı
    ]
)

# 4. ERKEN DURDURMA (Model en iyi halinde kalsın)
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# 5. EĞİTİMİ BAŞLAT (Hadi bismillah!)
print("🚀 Eğitim başlıyor...")
history = model.fit(
    X_train_res, y_train_res,
    epochs=50,
    batch_size=64,
    validation_data=(X_val_scaled, y_val),
    callbacks=[early_stop],
    verbose=1
)

🚀 Eğitim başlıyor...
Epoch 1/50
933/933 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - auc: 0.9741 - loss: 0.1523 - precision: 0.9063 - recall: 0.9094 - val_auc: 0.6072 - val_loss: 0.0386 - val_precision: 0.1537 - val_recall: 0.8514
Epoch 2/50
933/933 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - auc: 0.9950 - loss: 0.0608 - precision: 0.9771 - recall: 0.9527 - val_auc: 0.5940 - val_loss: 0.0312 - val_precision: 0.1641 - val_recall: 0.8514
Epoch 3/50
933/933 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - auc: 0.9975 - loss: 0.0426 - precision: 0.9806 - recall: 0.9701 - val_auc: 0.5803 - val_loss: 0.0305 - val_precision: 0.1512 - val_recall: 0.8378
Epoch 4/50
933/933 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - auc: 0.9984 - loss: 0.0301 - precision: 0.9829 - recall: 0.9832 - val_auc: 0.5913 - val_loss: 0.0256 - val_precision: 0.1854 - val_recall: 0.8243
Epoch 5/50
933/933 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - auc: 0.9989 - loss: 0.0225 - precision: 0.9861 - recall: 0.9925 - val_auc: 0.6012 - val_loss: 0.0228 - val_precision: 0.2186 

### 🧪 Kodunu Test Et

Orijinal dengesiz veri kümesinin (`X_test`, `y_test`) temsil edici bir örneğinde gerçek test performansınızın altında kalan sonuçları `precision` ve `recall` değişkenlerine kaydedin.

In [7]:
from sklearn.metrics import precision_score, recall_score

# 1. Modelin test seti üzerindeki tahminlerini alalım (Olasılık olarak döner: 0.0 - 1.0 arası)
y_pred_probs = model.predict(X_test_scaled)

# 2. Bu olasılıkları 0.5 eşiğine göre 0 veya 1'e çevirelim (Standart eşik)
y_pred = (y_pred_probs > 0.5).astype(int)

# 3. Hocanın istediği değişkenlere skorları kaydedelim
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

# 4. Sonuçları ekrana yazdıralım ki ne yaptığımızı görelim
print(f"--- FİNAL TEST SONUÇLARI ---")
print(f"Precision (Kesinlik): {precision:.4f}")
print(f"Recall (Duyarlılık): {recall:.4f}")

1336/1336 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
--- FİNAL TEST SONUÇLARI ---
Precision (Kesinlik): 0.4844
Recall (Duyarlılık): 0.8378


In [8]:
from nbresult import ChallengeResult

result = ChallengeResult('solution',
    precision=precision,
    recall=recall,
    fraud_number=len(y_test[y_test == 1]),
    non_fraud_number=len(y_test[y_test == 0]),
)

result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/macos/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/macos/Desktop/S18D3-S-Data-credit-card-challenge/tests
plugins: dash-4.0.0, anyio-4.8.0, typeguard-4.4.2
collecting ... collected 2 items

test_solution.py::TestSolution::test_is_score_good_enough PASSED         [ 50%]
test_solution.py::TestSolution::test_is_test_set_representative PASSED   [100%]

============================== 2 passed in 0.02s ===============================


💯 You can commit your code:

git add tests/solution.pickle

git commit -m 'Completed solution step'

git push origin master



## 🏁 İsteğe bağlı: Bu zorluk için Google'ın çözümünü okuyun
Bu oturumdaki tüm zorlukları tamamladığınız için tebrikler!

Son olarak, Google'ın kendi çözümünü doğrudan [Colab'da buradan](https://colab.research.google.com/github/tensorflow/docs/blob/master/site/en/tutorials/structured_data/imbalanced_data.ipynb) okuyun. 

İlginç teknikler ve en iyi uygulamaları keşfedeceksiniz.